In [1]:
%load_ext autoreload
%autoreload 2
import pandas as pd
from dfir_copilot.engine.query_engine import QueryEngine, QueryRejected

engine = QueryEngine("/workspace/data/processed/three_months.parquet", timeout_s=60, max_rows=1000)
print("dataset_sha256:", engine.dataset_sha256)

dataset_sha256: 128f5d7d57ce8ee30d2d22df2c6f43552963b55c44e3e560f6db8032aca9841a


In [2]:
display(engine.query("SELECT count(*) AS filas, count(DISTINCT user_id) AS identidades FROM logs").df())
display(engine.query("SELECT status_code, count(*) AS n FROM logs GROUP BY 1 ORDER BY n DESC").df())

,filas,identidades
0,4478619,35


,status_code,n
0,200,785553
1,400,739695
2,201,739226
3,304,736350
4,401,735324
5,202,733981
6,408,4254
7,406,4236


In [3]:
display(engine.query("""
    SELECT CAST(timestamp_utc AS DATE) AS dia, count(*) AS n
    FROM logs
    WHERE timestamp_utc >= TIMESTAMPTZ '2020-12-15 00:00:00+00'
    GROUP BY 1 ORDER BY 1
""").df())

,dia,n
0,2020-12-15,38448
1,2020-12-16,38448
2,2020-12-17,38448
3,2020-12-18,38448
4,2020-12-19,38448
5,2020-12-20,38448
6,2020-12-21,38448
7,2020-12-22,38448
8,2020-12-23,38448
9,2020-12-24,38448


In [4]:
for sql in ["DROP VIEW logs", "SELECT * FROM read_csv('/etc/passwd')", "SELECT 1; SELECT 2"]:
    try:
        engine.query(sql)
    except Exception as e:
        print(type(e).__name__, "->", str(e)[:90])

display(pd.DataFrame(engine.history)[["executed_at_utc", "status", "rows", "elapsed_ms", "sql"]])

QueryRejected -> Solo se permite SELECT; recibido: DROP
PermissionException -> Permission Error: Cannot access file "/etc/passwd" - file system operations are disabled b
QueryRejected -> Se permite exactamente una sentencia por llamada


,executed_at_utc,status,rows,elapsed_ms,sql
0,2026-10-04T22:42:56+00:00,ok,1,14,"SELECT count(*) AS filas, count(DISTINCT user_..."
1,2026-10-04T22:42:56+00:00,ok,8,12,"SELECT status_code, count(*) AS n FROM logs GR..."
2,2026-10-04T22:43:26+00:00,ok,17,48,\n SELECT CAST(timestamp_utc AS DATE) AS di...
3,2026-10-04T22:43:35+00:00,rejected,0,0,DROP VIEW logs
4,2026-10-04T22:43:35+00:00,error,0,2,SELECT * FROM read_csv('/etc/passwd')
5,2026-10-04T22:43:35+00:00,rejected,0,0,SELECT 1; SELECT 2
